# 📘 Semaine 13 — TP noté n°2 (ADC + Communication)

**Cours :** Microcontrôleurs STM32F103C6T6  
**Durée :** 4h30 (1h30 rappels + 1h30 TP noté + 1h30 remise rapport)  
**Enseignant :** ____________________  
**Étudiant :** ____________________  
**Binôme :** ____________________  
**Date :** ____________________

---

## 🎯 Objectifs de la semaine

1. **Réviser** l'ADC (S8), le DMA (S9) et les bus de communication (S10/S11).
2. **Démontrer** la maîtrise d'une chaîne d'acquisition complète en 1h30.
3. **Intégrer** ADC + DMA + TIMER + UART + (bonus I2C) dans un projet unique.
4. **Rédiger** un rapport technique final complet et rigoureux.

---

## 🗺️ Plan de la semaine

| Séquence | Durée | Contenu |
|---|---|---|
| **A — Rappels** | 1h30 | ADC, DMA, UART, I2C (fiches + mini-exercices) |
| **B — TP noté n°2** | 1h30 | Évaluation pratique en binôme |
| **C — Remise rapport** | 1h30 | Dépôt + feedback + révisions finales |

---

## 📊 Barème du TP noté n°2 (sur 20)

| Critère | Points |
|---|---|
| Configuration CubeMX (ADC + DMA + TIM + UART) | 4 |
| Code fonctionnel (acquisition + filtrage + seuil) | 8 |
| Mesures et précision | 3 |
| Communication UART (format + fiabilité) | 3 |
| **Bonus** : lecture capteur I2C | +2 |
| **Total** | **/20** |

---

# 🎓 PARTIE A — RAPPELS (1h30)

## 🔹 Fiche 1 — ADC (10 min)

### 📖 Caractéristiques

| Paramètre | Valeur |
|---|---|
| Résolution | 12 bits (4096 niveaux) |
| Canaux externes | 10 (IN0 à IN9) |
| V_ref | VDDA (typ. 3.3 V) |
| F_ADC max | 14 MHz |
| Temps conversion | (T_sample + 12.5) cycles |

### 📖 Formules

```
Code = round(V_in / V_ref × 4096)
V_in = Code × V_ref / 4096
LSB  = V_ref / 4096 ≈ 0.806 mV  (V_ref = 3.3 V)
```

### 📖 Code HAL

```c
HAL_ADCEx_Calibration_Start(&hadc1);
HAL_ADC_Start(&hadc1);
HAL_ADC_PollForConversion(&hadc1, 100);
uint16_t val = HAL_ADC_GetValue(&hadc1);
HAL_ADC_Stop(&hadc1);
```

---

## 🔹 Fiche 2 — DMA (10 min)

### 📖 Principes

- **DMA1** : 7 canaux
- **ADC1 → DMA1_Channel1**
- Modes : Normal / Circulaire
- Taille : 8, 16 ou 32 bits

### 📖 Callbacks HAL

```c
void HAL_ADC_ConvHalfCpltCallback(ADC_HandleTypeDef *hadc);  // demi-buffer
void HAL_ADC_ConvCpltCallback(ADC_HandleTypeDef *hadc);      // buffer complet
```

### 📖 Activation

```c
HAL_ADC_Start_DMA(&hadc1, (uint32_t*)buffer, TAILLE);
```

### 📖 Trigger matériel (timer)

TIM2 en TRGO → ADC démarre automatiquement à chaque update event.

---

## 🔹 Fiche 3 — UART (10 min)

### 📖 Format de trame 8N1

```
[START] [D0 D1 D2 D3 D4 D5 D6 D7] [STOP]
  1 bit       8 bits LSB first       1 bit
```

### 📖 Registre BRR (F_clk = 72 MHz)

```
USARTDIV = 72 000 000 / (16 × baud)
USART_BRR = mantisse[12 bits] + fraction[4 bits]

115200 → 0x271
```

### 📖 Code HAL

```c
HAL_UART_Transmit(&huart2, (uint8_t*)buf, len, 100);
HAL_UART_Receive_IT(&huart2, &rx, 1);

void HAL_UART_RxCpltCallback(UART_HandleTypeDef *huart);
```

---

## 🔹 Fiche 4 — I2C (10 min)

### 📖 Lignes et vitesses

| Standard | Fast | Fast+ |
|---|---|---|
| 100 kHz | 400 kHz | 1 MHz |

### 📖 Adresses typiques

| Adresse | Appareil |
|---|---|
| 0x3C | SSD1306 OLED |
| 0x48 | ADS1115 |
| 0x50 | AT24C32 EEPROM |
| 0x68 | MPU6050 |
| 0x76 | BMP280 |

### 📖 Code HAL

```c
HAL_I2C_Mem_Read(&hi2c1, 0x68 << 1, 0x75,
                 I2C_MEMADD_SIZE_8BIT, &data, 1, 100);
```

---

## 🔹 Mini-exercices de révision (30 min)

### ✍️ Exercice A — Conversions ADC

Pour un ADC 12 bits avec **V_ref = 3.3 V** :

| # | Tension | Code attendu |
|---|---|---|
| 1 | 0.0 V | ? |
| 2 | 0.5 V | ? |
| 3 | 1.0 V | ? |
| 4 | 1.65 V | ? |
| 5 | 2.0 V | ? |
| 6 | 2.5 V | ? |
| 7 | 3.0 V | ? |
| 8 | 3.3 V | ? |

Pour un seuil à **2.0 V**, quelle valeur de code doit-on comparer ?

In [ ]:
# Corrigé Exercice A
def tension_vers_code(v, v_ref=3.3, bits=12):
    return round(v / v_ref * (2 ** bits))

print(f"{'Tension':<10}{'Code':<10}")
print("-" * 25)
for v in [0.0, 0.5, 1.0, 1.65, 2.0, 2.5, 3.0, 3.3]:
    print(f"{v:<10.4f}{tension_vers_code(v)}")

print(f"\n🎯 Seuil 2.0 V → Code = {tension_vers_code(2.0)}")
print(f"   (soit 0x{tension_vers_code(2.0):03X} = {tension_vers_code(2.0):012b})")

### ✍️ Exercice B — Configuration DMA

Pour chaque scénario, indiquer la configuration DMA nécessaire :

| # | Scénario | Canal DMA | Mode | Largeur | Incrément |
|---|---|---|---|---|---|
| 1 | ADC1, buffer 100 uint16_t, continu | ? | ? | ? | ? |
| 2 | SPI1_RX, 64 octets, ponctuel | ? | ? | ? | ? |
| 3 | USART1_TX, 32 octets | ? | ? | ? | ? |

In [ ]:
# Corrigé Exercice B
configs = [
    (1, "ADC1 buffer 100 uint16_t continu",  "Canal 1", "Circulaire", "Half Word (16)", "Mémoire: Oui, Périph: Non"),
    (2, "SPI1_RX 64 octets ponctuel",        "Canal 2", "Normal",    "Byte (8)",      "Mémoire: Oui, Périph: Non"),
    (3, "USART1_TX 32 octets",               "Canal 4", "Normal",    "Byte (8)",      "Mémoire: Oui, Périph: Non"),
]
print(f"{'#':<4}{'Scénario':<35}{'Canal':<10}{'Mode':<12}{'Largeur':<16}{'Incrément'}")
print("-" * 100)
for c in configs:
    print(f"{c[0]:<4}{c[1]:<35}{c[2]:<10}{c[3]:<12}{c[4]:<16}{c[5]}")

### ✍️ Exercice C — Diagnostic

**L'ADC renvoie toujours 0. Causes possibles ?**
→ ...

**Le DMA ne remplit que le premier élément du buffer. Pourquoi ?**
→ ...

**Le PEC SMBus est incorrect. Que vérifier ?**
→ ...

**L'UART envoie des caractères aléatoires (garbage). Causes ?**
→ ...

### ✅ Corrigé Exercice C

| Problème | Causes possibles | Solutions |
|---|---|---|
| **ADC renvoie 0** | Broche non configurée en analogique / V_ref absent / ADON=0 / calibration manquante | Vérifier CubeMX, VDDA, `HAL_ADCEx_Calibration_Start()` |
| **DMA remplit 1 seul élément** | Mode Normal au lieu de Circulaire, increment memory désactivé | Activer Circular + Increment Memory |
| **PEC incorrect** | Mauvaise adresse, mauvais ordre des octets, bit R/W mal placé | Vérifier la séquence : `(addr << 1)`, commande, data |
| **UART garbage** | Baud rate différent, horloge non configurée, mauvais port COM | Vérifier BRR, USARTDIV, câblage TX/RX |

---

## 🔹 QCM de révision (10 min)

**1. La résolution de l'ADC du STM32F103 est :**  
A. 8 bits  B. 10 bits  C. 12 bits  D. 16 bits

**2. Sur quel canal DMA est mappé ADC1 ?**  
A. Canal 1  B. Canal 2  C. Canal 3  D. Canal 7

**3. Le callback appelé quand le buffer DMA est complètement rempli est :**  
A. `HAL_ADC_ConvHalfCpltCallback`  B. `HAL_ADC_ConvCpltCallback`  C. `HAL_DMA_CpltCallback`  D. `HAL_TIM_PeriodElapsedCallback`

**4. Sur le STM32F103C6T6, USART2 est sur les broches :**  
A. PA0/PA1  B. PA2/PA3  C. PA9/PA10  D. PB6/PB7

**5. Le baud rate de l'USART2 dans CubeMX par défaut est :**  
A. 9600  B. 19200  C. 115200  D. 460800

**6. I2C1 est par défaut sur les broches :**  
A. PA2/PA3  B. PB6/PB7  C. PA9/PA10  D. PB8/PB9

### ✅ Corrigé QCM

| Q | Rép. |
|---|---|
| 1 | **C — 12 bits** |
| 2 | **A — Canal 1** |
| 3 | **B — `HAL_ADC_ConvCpltCallback`** |
| 4 | **B — PA2/PA3** |
| 5 | **C — 115200** |
| 6 | **B — PB6/PB7** |

---

# 🛠️ PARTIE B — TP NOTÉ N°2 (1h30)

## 📋 Consignes générales

- **Durée : 1h30** (strict).
- **En binôme**. Un seul rendu par binôme.
- **Documents autorisés** : notes personnelles, datasheet, RM0008 (papier).
- **Sans échange entre binômes.** Aucun accès internet.
- **Livrables** : projet CubeIDE + rapport final + captures.

---

## 📄 Sujet du TP noté n°2

### 🎯 Objectif
Réaliser un **système d'acquisition et de communication** complet sur carte **STM32F103C6T6**.

### 📖 Cahier des charges

**1. Acquisition analogique**
- Signal sur **PA0** (ADC1_IN0).
- Résolution 12 bits, V_ref = 3.3 V.
- **Déclenchement par TIM2** à **1 kHz** (TRGO).
- **DMA circulaire** : buffer de **100 échantillons**.

**2. Traitement**
- Calcul d'une **moyenne sur 100 échantillons**.
- Conversion en tension : `V = moyenne × 3.3 / 4095`.
- **Détection de seuil** avec hystérésis :
  - Seuil haut = **2.0 V** (code ≈ 2482)
  - Seuil bas  = **1.8 V** (code ≈ 2234)
  - État transmis sur **PC13** (LED).

**3. Communication UART2**
- Baud rate : **115200**, format 8N1.
- Envoi à **10 Hz** (toutes les 100 ms) :
  ```
  moy=<code> V=<tension> etat=<0|1>
  ```
- Format compatible **SerialPlot** (pour tracé).

**4. Bonus : capteur I2C (+2 pts)**
- Si un capteur I2C est disponible (MPU6050, BMP280, etc.) :
  - Lire un registre (ex. température) toutes les secondes.
  - Ajouter la valeur au message UART.
- Si aucun capteur : simuler en lisant un registre et en envoyant la valeur.

### 📊 Barème détaillé

| Critère | Points | Détail |
|---|---|---|
| **Configuration CubeMX** | **4** | ADC + DMA + TIM2 + UART + GPIO + (I2C) |
| **Code fonctionnel** | **8** | Acquisition, moyenne, seuil, LED, UART |
| **Mesures et précision** | **3** | Écart avec multimètre, stabilité |
| **Communication UART** | **3** | Format lisible, régularité |
| **Bonus I2C** | **+2** | Lecture capteur |
| **Total** | **/20** | |

---

## 📖 Aide-mémoire autorisé

### Formules et valeurs

```
ADC 12 bits, V_ref = 3.3 V :
  Code = V_in × 4096 / 3.3
  V    = Code × 3.3 / 4096
  LSB  ≈ 0.806 mV

Seuil haut 2.0 V → Code ≈ 2482
Seuil bas  1.8 V → Code ≈ 2234

TIM2 à 1 kHz : PSC = 71, ARR = 999
Traitement   : 100 ms → toutes les 100 acquisitions

USART2 115200 : BRR = 0x271
```

### Squelette de code

```c
/* --- Variables --- */
#define TAILLE_BUFFER   100
#define SEUIL_HAUT      2482
#define SEUIL_BAS       2234

volatile uint16_t buffer_adc[TAILLE_BUFFER];
volatile uint8_t  drapeau_traitement = 0;
volatile uint8_t  etat_led = 0;

/* --- Calcul de la moyenne --- */
static uint16_t moyenne_buffer(void)
{
    uint32_t somme = 0;
    for (uint16_t i = 0; i < TAILLE_BUFFER; i++)
        somme += buffer_adc[i];
    return somme / TAILLE_BUFFER;
}

/* --- Traitement --- */
static void traiter(void)
{
    uint16_t moy = moyenne_buffer();
    float tension = moy * 3.3f / 4095.0f;

    if (moy > SEUIL_HAUT)      etat_led = 1;
    else if (moy < SEUIL_BAS)  etat_led = 0;

    HAL_GPIO_WritePin(GPIOC, GPIO_PIN_13,
                      etat_led ? GPIO_PIN_SET : GPIO_PIN_RESET);

    char buf[64];
    int n = snprintf(buf, sizeof(buf),
                     "%u %.4f %d\r\n", moy, tension, etat_led);
    HAL_UART_Transmit(&huart2, (uint8_t*)buf, n, 100);
}

/* --- Callbacks DMA --- */
void HAL_ADC_ConvHalfCpltCallback(ADC_HandleTypeDef *hadc)
{
    drapeau_traitement = 1;
}
void HAL_ADC_ConvCpltCallback(ADC_HandleTypeDef *hadc)
{
    drapeau_traitement = 1;
}

int main(void)
{
    HAL_Init();
    SystemClock_Config();
    MX_GPIO_Init();
    MX_DMA_Init();
    MX_ADC1_Init();
    MX_TIM2_Init();
    MX_USART2_UART_Init();

    HAL_ADCEx_Calibration_Start(&hadc1);
    HAL_TIM_Base_Start(&htim2);
    HAL_ADC_Start_DMA(&hadc1, (uint32_t*)buffer_adc, TAILLE_BUFFER);

    while (1)
    {
        if (drapeau_traitement)
        {
            drapeau_traitement = 0;
            traiter();
        }
    }
}
```

### Configuration CubeMX

| Périphérique | Configuration |
|---|---|
| PA0 | ADC1_IN0 |
| PC13 | GPIO_Output |
| PA2/PA3 | USART2 (Async, 115200) |
| PB6/PB7 | I2C1 (Fast 400 kHz) — bonus |
| TIM2 | PSC=71, ARR=999, TRGO=Update |
| ADC1 | Independent, 12 bits, DMA enabled, Trigger=Timer2 |
| DMA | Circular, Half Word, Increment Memory |

---

## 📝 Feuille de route (par binôme)

| Étape | Durée cible | Cocher |
|---|---|---|
| Créer le projet `TP_NOT2_NOM` | 5 min | ☐ |
| Configurer ADC + DMA + TIM2 + UART | 20 min | ☐ |
| Écrire le code (moyenne + seuil + UART) | 30 min | ☐ |
| Compiler, flasher, tester | 10 min | ☐ |
| Mesurer, comparer avec multimètre | 10 min | ☐ |
| Ajouter bonus I2C | 10 min | ☐ |
| Remplir le compte-rendu | 5 min | ☐ |
| **Total** | **1h30** | |

---

## 📋 Compte-rendu de TP noté n°2

**Nom 1 :** __________________  **Nom 2 :** __________________  
**Date :** __________________  **Groupe :** __________________

### 1. Configuration CubeMX (4 pts)

| Périphérique | Configuration |
|---|---|
| ADC1 | ... |
| DMA | ... |
| TIM2 | ... |
| USART2 | ... |
| GPIO | ... |
| I2C1 (bonus) | ... |

**Captures :** ☐ Pinout  ☐ Clock  ☐ ADC  ☐ DMA  ☐ TIM2  ☐ NVIC

### 2. Code fonctionnel (8 pts)

**Extrait du code ajouté :**
```c
// Coller ici le code (moyenne + seuil + UART)
```

### 3. Mesures et précision (3 pts)

| Position potentiomètre | Code ADC | Tension UART (V) | Tension multimètre (V) | Écart (mV) |
|---|---|---|---|---|
| 0 % | ... | ... | ... | ... |
| 25 % | ... | ... | ... | ... |
| 50 % | ... | ... | ... | ... |
| 75 % | ... | ... | ... | ... |
| 100 % | ... | ... | ... | ... |

**Écart maximal :** ... mV  
**Stabilité observée :** ±... mV

### 4. Communication UART (3 pts)

**Format envoyé :**
```
...
```

**Fréquence d'envoi mesurée :** ... Hz

### 5. Bonus I2C (2 pts)

**Capteur utilisé :** ...  
**Adresse :** 0x...  
**Registre lu :** 0x...  
**Valeur obtenue :** ...  
**Intégration dans le message UART :** oui / non

### 6. Difficultés rencontrées

- ...

### 7. Auto-évaluation

| Critère | Note estimée |
|---|---|
| Configuration CubeMX | /4 |
| Code fonctionnel | /8 |
| Mesures | /3 |
| Communication UART | /3 |
| Bonus I2C | /2 |
| **Total** | **/20** |

---

# 🎓 PARTIE C — REMISE DU RAPPORT FINAL (1h30)

## 📄 Rapport final — Structure attendue

### 1. Page de garde
- Titre du projet
- Noms des binômes
- Date

### 2. Introduction (½ page)
- Objectif du projet
- Contexte (cours de microcontrôleurs)

### 3. Architecture matérielle (1 page)
- Schéma de câblage
- Broches utilisées
- Alimentation

### 4. Configuration logicielle (1 page)
- CubeMX : captures + paramètres
- Choix techniques justifiés (PSC, ARR, CCR)

### 5. Code et algorithmes (2 pages)
- Structure du code
- Callbacks utilisés
- Traitement (moyenne, seuil)

### 6. Résultats et mesures (1 page)
- Tableau de mesures
- Comparaison multimètre
- Captures d'oscilloscope / terminal

### 7. Difficultés et solutions (½ page)
- Problèmes rencontrés
- Démarche de résolution

### 8. Conclusion (½ page)
- Bilan technique
- Améliorations possibles

### 9. Annexes
- Code complet (listing)
- Datasheets utilisées
- Bibliographie

---

## 📊 Grille d'évaluation du rapport (sur 10)

| Critère | 0-1 | 2 | 3 |
|---|---|---|---|
| **Structure** | Incomplète | Correcte | Complète et pro |
| **Justifications techniques** | Absentes | Partielles | Solides |
| **Résultats et mesures** | Peu nombreux | Présents | Riches et analysés |
| **Présentation** | Négligée | Correcte | Soignée |
| **Orthographe/grammaire** | Fautes | Quelques fautes | Aucune |

---

## 🎯 Corrigé type — Code complet (bonus I2C inclus)

In [ ]:
/* ============================================================
   CORRIGÉ TP NOTÉ N°2 — ADC + DMA + TIM2 + UART + I2C
   ============================================================ */

#include "main.h"
#include <stdio.h>

#define TAILLE_BUFFER   100
#define SEUIL_HAUT      2482   // ~2.0 V
#define SEUIL_BAS       2234   // ~1.8 V
#define ADRESSE_MPU     0x68
#define REG_WHO_AM_I    0x75

ADC_HandleTypeDef hadc1;
DMA_HandleTypeDef hdma_adc1;
TIM_HandleTypeDef htim2;
UART_HandleTypeDef huart2;
I2C_HandleTypeDef  hi2c1;

volatile uint16_t buffer_adc[TAILLE_BUFFER];
volatile uint8_t  drapeau = 0;
volatile uint8_t  etat_led = 0;

static uint16_t moyenne_buffer(void)
{
    uint32_t somme = 0;
    for (uint16_t i = 0; i < TAILLE_BUFFER; i++)
        somme += buffer_adc[i];
    return (uint16_t)(somme / TAILLE_BUFFER);
}

static uint8_t lire_whoami_i2c(void)
{
    uint8_t val = 0;
    if (HAL_I2C_Mem_Read(&hi2c1, ADRESSE_MPU << 1, REG_WHO_AM_I,
                         I2C_MEMADD_SIZE_8BIT, &val, 1, 100) == HAL_OK)
        return val;
    return 0xFF;   // erreur
}

static void traiter(void)
{
    uint16_t moy = moyenne_buffer();
    float tension = moy * 3.3f / 4095.0f;

    if (moy > SEUIL_HAUT)      etat_led = 1;
    else if (moy < SEUIL_BAS)  etat_led = 0;

    HAL_GPIO_WritePin(GPIOC, GPIO_PIN_13,
                      etat_led ? GPIO_PIN_SET : GPIO_PIN_RESET);

    uint8_t who = lire_whoami_i2c();
    char buf[80];
    int n;
    if (who != 0xFF) {
        n = snprintf(buf, sizeof(buf), "%u %.4f %d 0x%02X\r\n",
                     moy, tension, etat_led, who);
    } else {
        n = snprintf(buf, sizeof(buf), "%u %.4f %d\r\n",
                     moy, tension, etat_led);
    }
    HAL_UART_Transmit(&huart2, (uint8_t*)buf, n, 100);
}

void HAL_ADC_ConvHalfCpltCallback(ADC_HandleTypeDef *hadc)
{
    drapeau = 1;
}
void HAL_ADC_ConvCpltCallback(ADC_HandleTypeDef *hadc)
{
    drapeau = 1;
}

int main(void)
{
    HAL_Init();
    SystemClock_Config();
    MX_GPIO_Init();
    MX_DMA_Init();
    MX_ADC1_Init();
    MX_TIM2_Init();
    MX_USART2_UART_Init();
    MX_I2C1_Init();

    HAL_ADCEx_Calibration_Start(&hadc1);
    HAL_TIM_Base_Start(&htim2);
    HAL_ADC_Start_DMA(&hadc1, (uint32_t*)buffer_adc, TAILLE_BUFFER);

    while (1)
    {
        if (drapeau)
        {
            drapeau = 0;
            traiter();
        }
    }
}

---

## 🐍 Simulation Python — Traitement côté PC (SerialPlot)

In [ ]:
# ============================================================
# Script Python côté PC — réception UART + tracé
# ============================================================

def script_pc_acquisition():
    """
    Script à exécuter sur le PC.
    Nécessite : pip install pyserial matplotlib
    """
    code = '''
import serial
import time
import matplotlib.pyplot as plt

ser = serial.Serial('COM3', 115200, timeout=1)  # Adapter le port
time.sleep(2)

codes     = []
tensions  = []
etats     = []

N_POINTS = 200

print("Acquisition en cours... Ctrl+C pour arrêter")
try:
    while len(codes) < N_POINTS:
        ligne = ser.readline().decode('ascii', errors='ignore').strip()
        if not ligne:
            continue
        parties = ligne.split()
        if len(parties) >= 3:
            codes.append(int(parties[0]))
            tensions.append(float(parties[1]))
            etats.append(int(parties[2]))
            print(f"{len(codes):>4}  Code={parties[0]}  V={parties[1]}  Etat={parties[2]}")
except KeyboardInterrupt:
    pass

ser.close()

# Tracé
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6))
ax1.plot(tensions)
ax1.set_ylabel('Tension (V)')
ax1.axhline(2.0, color='r', linestyle='--', label='Seuil haut')
ax1.axhline(1.8, color='orange', linestyle='--', label='Seuil bas')
ax1.legend()
ax1.grid()
ax2.step(range(len(etats)), etats)
ax2.set_ylabel('Etat LED')
ax2.set_xlabel('Échantillon')
ax2.grid()
plt.tight_layout()
plt.show()
'''
    return code

print("💻 Script Python à exécuter sur le PC :\n")
print(script_pc_acquisition())

print("\n📌 Notes :")
print("  → Adapter le port COM (Windows) ou /dev/ttyUSB0 (Linux)")
print("  → SerialPlot est une alternative graphique simple")
print("  → Format envoyé compatible : 'code tension etat\\r\\n'")

---

## 🔍 Erreurs fréquentes et solutions

| Erreur | Cause probable | Solution |
|---|---|---|
| ADC renvoie toujours 0 | Broche non configurée en analogique / Calibration manquante | Vérifier CubeMX et appeler `HAL_ADCEx_Calibration_Start()` |
| DMA remplit 1 seul élément | Increment Memory désactivé | Activer Increment Memory |
| Buffer non mis à jour | Mode Normal au lieu de Circular | Changer en Circular |
| Valeurs instables | Bruit / Impédance source trop élevée | Augmenter sampling time (41.5 ou 239.5 cycles) |
| UART garbage | Baud rate incorrect | Vérifier BRR = 0x271 pour 115200 |
| I2C ne répond pas | Pull-up absents | Ajouter 4.7 kΩ sur SDA et SCL |
| Callback DMA jamais appelé | NVIC DMA non activé | Cocher "DMA1 channel1 global interrupt" |
| Fréquence d'envoi UART trop lente | HAL_UART_Transmit bloquant | Réduire la taille ou utiliser DMA UART |

---

## 📊 Grille d'auto-évaluation finale

### Configuration CubeMX (4 pts)

- [ ] PA0 configuré en ADC1_IN0
- [ ] ADC1 en mode Independent + DMA enabled
- [ ] Trigger = Timer 2 TRGO
- [ ] DMA1 canal 1, mode Circulaire, Half Word, Increment Memory
- [ ] TIM2 : PSC=71, ARR=999 (1 kHz)
- [ ] USART2 : 115200, 8N1
- [ ] PC13 configuré en GPIO_Output
- [ ] I2C1 activé (bonus)

### Code fonctionnel (8 pts)

- [ ] `HAL_ADCEx_Calibration_Start()` appelé
- [ ] `HAL_TIM_Base_Start()` appelé
- [ ] `HAL_ADC_Start_DMA()` appelé avec le bon buffer
- [ ] Callbacks DMA implémentés (Half + Cplt)
- [ ] Moyenne sur 100 échantillons calculée
- [ ] Seuil haut/bas avec hystérésis
- [ ] LED PC13 mise à jour
- [ ] Message UART formaté correctement

### Mesures et précision (3 pts)

- [ ] Comparaison avec multimètre effectuée
- [ ] Écart < 50 mV
- [ ] Stabilité observée

### Communication UART (3 pts)

- [ ] Format `moy tension etat` respecté
- [ ] Envoi régulier à ~10 Hz
- [ ] Compatible SerialPlot

### Bonus I2C (2 pts)

- [ ] Capteur détecté (scan I2C)
- [ ] Registre lu et intégré au message

---

# 🎯 PRÉPARATION S14 — RÉVISION GÉNÉRALE

## 📋 Programme de la semaine 14

- **Synthèse générale** : architecture → GPIO → EXTI → TIMER → PWM → ADC → COM
- **Examen blanc** : QCM 40 questions + 1 exercice de code
- **Correction + Q&R**

## 📖 À préparer pour S14

1. **Relire** tous les notebooks S1 à S13 (récapitulatifs).
2. **Refaire** les QCM formatifs.
3. **S'entraîner** sur les calculs PSC/ARR/CCR.
4. **Vérifier** la maîtrise des registres clés : CRL, CRH, BSRR, IMR, RTSR, PR, PSC, ARR, CCR, BRR.
5. **Identifier** les points faibles personnels et les revoir.

## 🗺️ Carte mentale de révision

```
STM32F103C6T6
├── Architecture
│   ├── ARM Cortex-M3 (pipeline, registres, NVIC)
│   └── Bus AHB/APB1/APB2 + mappe mémoire
├── GPIO
│   ├── CRL/CRH (config)
│   ├── IDR/ODR/BSRR
│   └── HAL/LL/Registres
├── EXTI/NVIC
│   ├── AFIO_EXTICR
│   ├── IMR/RTSR/FTSR/PR
│   └── Priorités
├── TIMER
│   ├── PSC/ARR/CNT
│   ├── IT update
│   └── PWM (CCR)
├── ADC
│   ├── 12 bits, 10 canaux
│   ├── Polling/IT/DMA
│   └── Trigger timer
└── Communication
    ├── UART (BRR)
    ├── I2C (adressage, ACK)
    ├── SPI (CPOL/CPHA)
    └── SMBus (PEC)
```

---

## 📊 Bilan de la semaine 13

| Item | Statut |
|---|---|
| Révisions A (fiches + exercices) | ☐ |
| TP noté n°2 (1h30) | ☐ |
| Compte-rendu rempli | ☐ |
| Rapport final déposé | ☐ |
| Auto-évaluation remplie | ☐ |
| Préparation S14 | ☐ |

### 📈 Note estimée du TP noté n°2 : ___ / 20

| Score | Interprétation |
|---|---|
| 16–20 | ✅ Excellent — maîtrise complète |
| 12–15 | ✅ Bien — quelques points à revoir |
| 8–11 | ⚠️ Passable — révisions nécessaires |
| < 8 | ❌ Insuffisant — revoir S8 et S9 |

---
# 📚 RESSOURCES Semaine 13

### Documents officiels
- 📄 **RM0008** — chapitres 10 (DMA), 11 (ADC), 26 (I2C), 27 (USART)
- 📄 **Datasheet STM32F103x6**
- 📄 **AN2834** — Getting the best ADC accuracy
- 📄 **AN3116** — ADC modes
- 📄 **UM1850** — HAL documentation

### Notebooks de référence
- 📓 S8 — ADC (principes)
- 📓 S9 — ADC + DMA
- 📓 S10 — I2C et SPI
- 📓 S11 — UART/USART et SMBus

### Outils
- **STM32CubeIDE**
- **STM32CubeMX**
- **ST-Link V2**
- **Multimètre** (vérification tensions)
- **Terminal série** (PuTTY, minicom)
- **SerialPlot** ou **Python/matplotlib** pour tracer

### Bonnes pratiques — Récapitulatif

- Toujours **calibrer** l'ADC avant usage
- Utiliser **DMA** au-delà de 1 kHz
- **Trigger timer** pour acquisition périodique précise
- Placer les buffers en **`volatile`**
- **Double buffering** pour ne pas bloquer
- **Filtrage** : moyenne, médiane, IIR
- **Hystérésis** pour les seuils
- **ISR courtes** : poser un flag, traiter hors ISR
- **UART** : préférer réception IT/DMA en production
- **I2C** : pull-up obligatoires sur SDA/SCL

---

**Fin du notebook — Semaine 13** ✨